# Sprint 4 Live — Rehearse the integration evidence

Use this separate Live notebook for LS13–LS16. The Lovable project is the interface students build; this notebook makes real requests to the provisioned FieldCare service so the class can inspect the caller contract, stream, deterministic clarification, and rejection boundary.

A direct notebook request does not establish that the Lovable client or Edge Function is wired correctly. For the UI path, use the actual Lovable preview and the service request ID. Eligible requests use ChromaDB, LangGraph, and OpenRouter; no cell substitutes a canned answer.

## Live-session sequence

1. Build or adapt the bounded form in Lovable and inspect the generated code against the supplied client adapter.
2. Use the request cell to inspect one actual service stream and its terminal event.
3. Compare missing-context clarification, invalid input, and missing authentication as separate boundaries.
4. Use the observed statuses, event fields, and request ID as evidence for the end-to-end failure report and README/API handoff.

## 1. Set up the course connection once

The facilitator supplies a reachable HTTPS origin and caller key. Add them to Colab Secrets as `FIELDCARE_SERVICE_URL` and `FIELDCARE_CALLER_KEY`. The OpenRouter key remains on the FieldCare service; this caller notebook must not receive it.

In [ ]:
# One-time setup: use the course-provisioned reachable Sprint 3 service.
import json
import time
from urllib.parse import urlsplit

import requests
from google.colab import userdata

SERVICE_URL = (userdata.get("FIELDCARE_SERVICE_URL") or "").strip().rstrip("/")
CALLER_KEY = (userdata.get("FIELDCARE_CALLER_KEY") or "").strip()

parts = urlsplit(SERVICE_URL)
if (parts.scheme != "https" or not parts.netloc or parts.path not in ("", "/")
        or parts.username or parts.password or parts.query or parts.fragment):
    raise RuntimeError("Add the course HTTPS service origin, without a path or credentials, as FIELDCARE_SERVICE_URL.")
if not CALLER_KEY:
    raise RuntimeError("Add the course caller credential to Colab Secrets as FIELDCARE_CALLER_KEY.")

print("Course service settings loaded; credential values are hidden.")


## 2. Define one bounded observer

This function sends HTTP requests with finite connect/read timeouts. It runs no local FastAPI server, so the notebook cell can return after the response or a timeout. Stream text is shown only when it arrives from the actual service.

In [ ]:
def call_fieldcare(payload, *, authenticate=True, label="request"):
    """Call the real Sprint 3 API; print only the response produced by this run."""
    headers = {"Accept": "application/x-ndjson, application/json"}
    if authenticate:
        headers["X-API-Key"] = CALLER_KEY
    record = {
        "label": label,
        "http_status": None,
        "request_id": None,
        "content_type": None,
        "application_status": None,
        "citations": [],
        "terminal_event": None,
        "provider_delta_count": 0,
        "answer": "",
    }
    try:
        with requests.post(
            f"{SERVICE_URL}/v1/diagnose-stream",
            json=payload,
            headers=headers,
            stream=True,
            timeout=(5, 45),
        ) as response:
            record["http_status"] = response.status_code
            record["request_id"] = response.headers.get("X-Request-ID")
            record["content_type"] = response.headers.get("Content-Type", "")
            if response.status_code != 200:
                try:
                    record["error_detail"] = response.json().get("detail", "")
                except (ValueError, AttributeError):
                    record["error_detail"] = "Response body was not JSON."
                print(label, "HTTP", record["http_status"], record["error_detail"])
                return record

            if "application/x-ndjson" not in record["content_type"]:
                body = response.json()
                record["application_status"] = body.get("status")
                record["citations"] = body.get("citations", [])
                record["answer"] = body.get("answer", "")
                print(label, "HTTP", response.status_code, "application status:", record["application_status"])
                print("Actual JSON response:", body)
                return record

            deadline = time.monotonic() + 90
            for line in response.iter_lines(decode_unicode=True):
                if time.monotonic() > deadline:
                    raise TimeoutError("No terminal stream event arrived within 90 seconds.")
                if not line:
                    continue
                event = json.loads(line)
                event_type = event.get("type")
                if event_type == "metadata":
                    record["request_id"] = event.get("request_id", record["request_id"])
                    record["application_status"] = event.get("status")
                    record["citations"] = event.get("citations", [])
                    print("metadata:", {key: event.get(key) for key in ("status", "citations", "mode", "request_id")})
                elif event_type == "delta":
                    piece = event.get("text", "")
                    record["answer"] += piece
                    record["provider_delta_count"] += 1
                    print(piece, end="", flush=True)
                elif event_type == "usage":
                    print("\nusage:", {key: event.get(key) for key in ("model", "tokens")})
                elif event_type in ("complete", "error"):
                    record["terminal_event"] = event_type
                    print("\nterminal event:", event)
            if record["terminal_event"] is None:
                print("\nThe connection ended without a terminal event; treat this as incomplete.")
    except (requests.RequestException, TimeoutError, ValueError) as error:
        record["failure"] = type(error).__name__
        print(label, "request did not complete:", type(error).__name__)
    return record


## 3. Test the caller boundaries

Predict the outcomes before running the cases. The first request omits the caller key and should stop at authentication. The second is valid against the schema but lacks optional equipment context, so the service should return its actual deterministic clarification. The third omits required `question` and should be rejected by schema validation.

In [ ]:
unauthenticated = call_fieldcare(
    {"question": "Which filter checks are documented?", "equipment_id": "EQ-FC-1002"},
    authenticate=False,
    label="missing caller credential",
)
clarification = call_fieldcare(
    {"question": "Which airflow checks are documented?"},
    label="valid input missing optional context",
)
invalid = call_fieldcare({}, label="missing required question")

print("Observed HTTP outcomes:", {
    "unauthenticated": unauthenticated["http_status"],
    "clarification": clarification["http_status"],
    "schema-invalid": invalid["http_status"],
})

## 4. Make one real provider-backed request

Run this only after reviewing the earlier cases. This one supported request can use provider quota for retrieval embeddings and generation. Inspect the actual output and citations; model wording varies. Keep an error as an error if the provider or connection fails.

In [ ]:
supported = {
    "question": "Which filter and airflow checks are documented?",
    "equipment_id": "EQ-FC-1002",
}
supported_observation = call_fieldcare(supported, label="supported model-backed request")

## 5. Write the failure record from observed evidence

A failure report separates an observation from a hypothesis. Use the records above for the API path, then repeat the workflow in Lovable and capture the UI request ID and matching safe service event. Do not claim the notebook calls exercised the browser or connector.

In [ ]:
attempts = [unauthenticated, clarification, invalid, supported_observation]
for attempt in attempts:
    print({key: attempt.get(key) for key in (
        "label", "http_status", "request_id", "application_status",
        "terminal_event", "provider_delta_count", "failure"
    )})

## 6. Export a sanitized handoff

The file omits caller keys, request bodies, generated text, and raw headers. Add the Lovable UI request ID and your failure narrative in the README or the course handoff template; those must come from your actual preview run.

In [ ]:
from datetime import datetime, timezone
from google.colab import files

live_evidence = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook_is_direct_api_caller": True,
    "attempts": [
        {key: attempt.get(key) for key in (
            "label", "http_status", "request_id", "content_type",
            "application_status", "citations", "terminal_event",
            "provider_delta_count", "failure"
        )}
        for attempt in attempts
    ],
    "ui_evidence_required_separately": True,
}
evidence_path = "/content/sprint-4-live-api-evidence.json"
with open(evidence_path, "w", encoding="utf-8") as stream:
    json.dump(live_evidence, stream, indent=2)
files.download(evidence_path)